# Build the base aspect vocabulary

This notebook is part of the public reproducibility workflow for the study.

Public workflow stage: 03  
Purpose: Merge extraction results, count review-level terms, and build the base vocabulary.

Source notebook:
- `主表及附属表构建.ipynb` (cells 1–9 and 33–39, formal base-vocabulary branch)

Inputs: `data/work/aspect_responses/results_part_*.jsonl`, matching sentence parts, and `data/sample/sample_papers.csv` (sample provided later).  
Outputs: `data/work/review_aspect_extract.parquet`, `data/work/review_term_counts.parquet`, and `data/work/tables_and_vocab/`.


In [ ]:
from pathlib import Path
import sys

# Jupyter may start in the repository root or its notebooks directory.
_candidates = (Path.cwd().resolve(), Path.cwd().resolve().parent)
_root = next((p for p in _candidates if (p / "src" / "project_paths.py").is_file()), None)
if _root is None:
    raise RuntimeError("Start Jupyter from the repository root or notebooks directory")
sys.path.insert(0, str(_root))
from src.project_paths import SAMPLE_DATA_DIR, WORK_DATA_DIR, CONFIG_DIR, project_path

try:
    import yaml
except ImportError as exc:
    raise ImportError("Install PyYAML to read config/config.example.yaml") from exc
with (CONFIG_DIR / "config.example.yaml").open(encoding="utf-8") as _f:
    config = yaml.safe_load(_f)
sample_dir = project_path(config["paths"]["sample_data"])
work_dir = project_path(config["paths"]["work_data"])

import json
import re
import unicodedata
from collections import defaultdict
import pandas as pd
from tqdm.auto import tqdm
from nltk.stem import WordNetLemmatizer

response_dir = work_dir / "aspect_responses"
sentence_dir = work_dir / "sentence_parts"
table_dir = work_dir / "tables_and_vocab"
paper_path = sample_dir / "sample_papers.csv"
if not paper_path.is_file():
    raise FileNotFoundError(f"Paper sample is not available yet: {paper_path}")
table_dir.mkdir(parents=True, exist_ok=True)


## Parse JSONL and merge sentence-part metadata

Malformed lines are counted and skipped. Repeated `(paper_id, review_id)` results in one part retain the last line, matching the formal source branch.


In [ ]:
def parse_part_idx(name):
    match = re.search(r"(?:part[_-])(\d{5})", name)
    return match.group(1) if match else None

def iter_jsonl(path):
    with path.open(encoding="utf-8") as stream:
        for line_no, line in enumerate(stream, 1):
            if line.strip():
                try:
                    yield line_no, json.loads(line)
                except json.JSONDecodeError as exc:
                    yield line_no, {"__jsonl_parse_error__": str(exc)}

def normalize_aspects_json_string(value):
    if value is None:
        return None, "aspects_is_none"
    if isinstance(value, str):
        if not value.strip():
            return None, "aspects_empty_string"
        try:
            value = json.loads(value.strip())
        except json.JSONDecodeError as exc:
            return value.strip(), f"aspects_json_parse_error: {exc}"
    if not isinstance(value, dict):
        return json.dumps(value, ensure_ascii=False), "aspects_not_dict_but_serialized"
    normalized = {}
    for key, item in value.items():
        normalized[str(key)] = (None if item is None else item.strip() if isinstance(item, str)
                                else json.dumps(item, ensure_ascii=False))
    ordered = {key: normalized[key] for key in sorted(
        normalized, key=lambda key: (0, int(key)) if key.isdigit() else (1, key))}
    return json.dumps(ordered, ensure_ascii=False), None

def load_results_part_latest(path):
    rows, bad_lines, warnings = [], 0, 0
    for line_no, item in iter_jsonl(path):
        if not isinstance(item, dict) or "__jsonl_parse_error__" in item:
            bad_lines += 1
            continue
        if item.get("paper_id") is None or item.get("review_id") is None:
            bad_lines += 1
            continue
        aspects, warning = normalize_aspects_json_string(item.get("aspects"))
        warnings += warning is not None
        rows.append(dict(paper_code=str(item["paper_id"]), review_id=str(item["review_id"]),
                         aspects_by_sent_json=aspects, _line_no=line_no))
    columns = ["paper_code", "review_id", "aspects_by_sent_json"]
    if not rows:
        return pd.DataFrame(columns=columns), dict(rows_read=0, bad_lines=bad_lines,
                                                    dup_overwrites=0, aspects_parse_warn=warnings)
    result = pd.DataFrame(rows).sort_values("_line_no", kind="mergesort")
    result = result.drop_duplicates(["paper_code", "review_id"], keep="last")
    result = result.drop(columns="_line_no").reset_index(drop=True)
    return result, dict(rows_read=len(rows), bad_lines=bad_lines,
                        dup_overwrites=len(rows) - len(result), aspects_parse_warn=warnings)

def load_review_meta_from_part(path, part_idx):
    frame = pd.read_parquet(path, columns=["review_id", "paper_code", "reviewer_tag", "year", "sent_idx"])
    for column in ("review_id", "paper_code", "reviewer_tag"):
        frame[column] = frame[column].astype(str)
    frame["year"] = pd.to_numeric(frame["year"], errors="coerce").astype("Int64")
    meta = frame.groupby(["paper_code", "review_id"], as_index=False).agg(
        reviewer_tag=("reviewer_tag", "first"), year=("year", "first"),
        n_sent_total=("sent_idx", "size"))
    meta["part_idx"] = part_idx
    return meta[["review_id", "paper_code", "reviewer_tag", "part_idx", "year", "n_sent_total"]]

result_files = sorted(response_dir.glob("results_part_*.jsonl"))
if not result_files:
    raise FileNotFoundError(f"No aspect-result JSONL files were found in {response_dir}")
parts, part_stats = [], []
for result_path in tqdm(result_files, desc="Merge aspect results"):
    part_idx = parse_part_idx(result_path.name)
    if part_idx is None:
        continue
    sentence_path = sentence_dir / f"part-{part_idx}.parquet"
    if not sentence_path.is_file():
        print(f"Skipping result without a matching sentence part: {result_path.name}")
        continue
    meta = load_review_meta_from_part(sentence_path, part_idx)
    result, stats = load_results_part_latest(result_path)
    merged = meta.merge(result, on=["paper_code", "review_id"], how="left", validate="one_to_one")
    merged["n_sent_total"] = pd.to_numeric(merged["n_sent_total"], errors="coerce").astype("Int32")
    merged["year"] = pd.to_numeric(merged["year"], errors="coerce").astype("Int64")
    parts.append(merged)
    part_stats.append(dict(part_idx=part_idx, result_file=result_path.name,
                           meta_reviews=len(meta), missing_aspects_after_merge=int(merged["aspects_by_sent_json"].isna().sum()),
                           **stats))
if not parts:
    raise RuntimeError("No matching result and sentence parts were found")
review_aspect_extract = pd.concat(parts, ignore_index=True).sort_values(
    ["part_idx", "paper_code", "review_id"], kind="mergesort").reset_index(drop=True)
review_aspect_extract.to_parquet(work_dir / "review_aspect_extract.parquet", index=False)
pd.DataFrame(part_stats).to_csv(table_dir / "review_aspect_extract_build_stats.csv", index=False)
print(f"Saved {len(review_aspect_extract)} review-aspect rows")


## Count terms within each review

The raw counter preserves case. `count_in_review` counts every term occurrence; `n_sent_hit` counts distinct sentence keys containing that term.


In [ ]:
def safe_json_loads(value):
    if value is None or not isinstance(value, (dict, str)):
        return None, "missing_or_unsupported_aspects"
    if isinstance(value, dict):
        return value, None
    try:
        result = json.loads(value.strip())
    except json.JSONDecodeError as exc:
        return None, f"json_parse_error:{exc}"
    return (result, None) if isinstance(result, dict) else (None, "json_not_dict")

def split_aspect_terms(value):
    if value is None:
        return []
    text = str(value).strip()
    if not text or text.lower() in {"none", "null", "nan"}:
        return []
    # The two non-ASCII separators are source parsing literals, not commentary.
    text = text.replace("，", ",").replace("；", ",").replace(";", ",")
    return [item for item in (part.strip() for part in text.split(","))
            if item and item.lower() not in {"none", "null", "nan"}]

def normalize_term_raw(term):
    return re.sub(r"\s+", " ", str(term)).strip()

def build_review_term_counts(review_frame):
    rows, parse_errors, zero_terms, occurrences = [], 0, 0, 0
    for record in tqdm(review_frame.itertuples(index=False), total=len(review_frame), desc="Count review terms"):
        aspects, error = safe_json_loads(record.aspects_by_sent_json)
        if error:
            parse_errors += 1
            continue
        counter, hits = defaultdict(int), defaultdict(set)
        for sent_key, value in aspects.items():
            for term in split_aspect_terms(value):
                raw = normalize_term_raw(term)
                if raw:
                    counter[raw] += 1
                    hits[raw].add(str(sent_key))
                    occurrences += 1
        if not counter:
            zero_terms += 1
        for term_raw, count in counter.items():
            rows.append(dict(review_id=str(record.review_id), paper_code=str(record.paper_code),
                             year=record.year, term_raw=term_raw,
                             count_in_review=count, n_sent_hit=len(hits[term_raw])))
    columns = ["review_id", "paper_code", "year", "term_raw", "count_in_review", "n_sent_hit"]
    result = pd.DataFrame(rows, columns=columns)
    if len(result):
        result["year"] = pd.to_numeric(result["year"], errors="coerce").astype("Int64")
        result["count_in_review"] = result["count_in_review"].astype("Int32")
        result["n_sent_hit"] = result["n_sent_hit"].astype("Int16")
        result = result.sort_values(["paper_code", "review_id", "term_raw"], kind="mergesort").reset_index(drop=True)
    stats = dict(n_reviews_input=len(review_frame), n_reviews_parse_error_skipped=parse_errors,
                 n_reviews_zero_terms=zero_terms, n_rows_review_term_counts=len(result),
                 total_terms_before_group_within_review=occurrences,
                 n_unique_terms_global=result["term_raw"].nunique())
    return result, stats

review_term_counts, term_stats = build_review_term_counts(review_aspect_extract)
review_term_counts.to_parquet(work_dir / "review_term_counts.parquet", index=False)
(table_dir / "review_term_counts_build_stats.json").write_text(json.dumps(term_stats, indent=2), encoding="utf-8")
print(f"Saved {len(review_term_counts)} review-term rows")


## Adapt paper metadata and write mapping tables

The public CSV uses the same information contract as the formal paper sheet. Original non-English source columns remain explicit adapter keys; all downstream columns are English.


In [ ]:
# ALLOWED_SOURCE_SCHEMA_LITERAL: formal NCsubject source-column names only.
COLUMN_MAP = {"论文编码": "paper_code", "发表时间": "pub_date_raw",
              "主题": "topic", "标题": "title", "subject": "subject_code"}
papers = pd.read_csv(paper_path).rename(columns=COLUMN_MAP)
if "pub_date_raw" not in papers and "pub_date" in papers:
    papers = papers.rename(columns={"pub_date": "pub_date_raw"})
required = {"paper_code", "subject_code", "pub_date_raw"}
if missing := required.difference(papers.columns):
    raise KeyError(f"Paper input is missing columns: {sorted(missing)}")
selected = ["paper_code", "subject_code", "pub_date_raw"] + [
    column for column in ("topic", "title") if column in papers.columns]
paper_subject_map = papers[selected].copy()
paper_subject_map["paper_code"] = paper_subject_map["paper_code"].astype(str).str.strip()
paper_subject_map["subject_code"] = paper_subject_map["subject_code"].astype(str).str.strip()
paper_subject_map["pub_date_raw"] = paper_subject_map["pub_date_raw"].apply(
    lambda value: "" if pd.isna(value) else str(value))
paper_subject_map["pub_date"] = pd.to_datetime(paper_subject_map["pub_date_raw"], errors="coerce")
paper_subject_map["year"] = paper_subject_map["pub_date"].dt.year
paper_subject_map.to_parquet(table_dir / "paper_subject_map.parquet", index=False)

paper_meta_source = paper_subject_map.sort_values(["paper_code", "pub_date"]).copy()
if "topic" not in paper_meta_source:
    paper_meta_source["topic"] = None
if "title" not in paper_meta_source:
    paper_meta_source["title"] = None
paper_meta = paper_meta_source.groupby("paper_code", as_index=False).agg(
    pub_date=("pub_date", "min"), year=("year", "min"),
    subject_codes=("subject_code", lambda values: sorted(set(
        value for value in values if isinstance(value, str) and value.strip()))),
    topic_list=("topic", lambda values: sorted(set(
        str(value).strip() for value in values if pd.notna(value)))),
    title=("title", lambda values: values.dropna().iloc[0] if len(values.dropna()) else None))
paper_meta.to_parquet(table_dir / "paper_meta.parquet", index=False)

review_paper_map = review_term_counts[["review_id", "paper_code"]].drop_duplicates().copy()
if review_term_counts["year"].notna().any():
    year_map = review_term_counts[["review_id", "year"]].dropna().groupby("review_id")["year"].agg(
        lambda values: int(pd.Series(values).mode().iloc[0]) if len(pd.Series(values).mode()) else int(values.iloc[0]))
    review_paper_map = review_paper_map.merge(year_map.rename("review_year"), on="review_id", how="left")
else:
    review_paper_map["review_year"] = float("nan")
review_paper_map.to_parquet(table_dir / "review_paper_map.parquet", index=False)
print("Saved paper and review mapping tables")


## Normalize terms and calculate the base vocabulary

This retains the formal source's light WordNet lemmatization and weighted frequency statistics. It does not apply later filtering, manual categories, or TF-IDF anchors. WordNet data must be installed locally before execution; this notebook does not download it.


In [ ]:
HYPHEN_VARIANTS = ["\u2010", "\u2011", "\u2012", "\u2013", "\u2014", "\u2015", "\u2212", "\ufe63", "\uff0d"]
ZERO_WIDTH = ["\u200b", "\u200c", "\u200d", "\ufeff", "\u2060"]
NBSP = ["\u00a0", "\u202f", "\u2007"]
ACRONYM_RE = re.compile(r"^[A-Z0-9]{2,}$")
lemmatizer = WordNetLemmatizer()

def clean_text(value):
    if value is None:
        return ""
    text = unicodedata.normalize("NFKC", str(value))
    for character in ZERO_WIDTH:
        text = text.replace(character, "")
    for character in NBSP:
        text = text.replace(character, " ")
    for character in HYPHEN_VARIANTS:
        text = text.replace(character, "-")
    return re.sub(r"\s+", " ", text).strip().strip(" \t\r\n\"'“”‘’`.,;:()[]{}")

def remove_diacritics(text):
    text = unicodedata.normalize("NFKD", text)
    return unicodedata.normalize("NFKC", "".join(
        character for character in text if not unicodedata.combining(character)))

def tokenize_phrase(text):
    return [token for token in re.split(r"\s+", text.replace("-", " ")) if token]

def smart_title(tokens):
    return " ".join(token if ACRONYM_RE.match(token) else token[0].upper() + token[1:]
                    for token in tokens if token)

def guess_pos(token):
    return "v" if token.endswith(("ing", "ed")) and len(token) >= 4 else "n"

def nltk_lemma_token(token):
    if ACRONYM_RE.match(token):
        return token
    token = re.sub(r"[^a-z0-9]+", "", remove_diacritics(token).lower())
    if len(token) <= 1:
        return ""
    try:
        lemma = lemmatizer.lemmatize(token, pos=guess_pos(token))
    except LookupError:
        lemma = token  # Offline no-WordNet fallback; never download a corpus
    return lemma if len(lemma) > 1 else ""

def normalize_term(term_raw):
    raw = clean_text(term_raw)
    if not raw:
        return dict(term_raw=term_raw, term_clean="", term_lemma="", term_norm="", drop_reason="empty")
    raw2 = re.sub(r"[0-9]+", "", raw)
    raw2 = re.sub(r"[^A-Za-z\- ]+", " ", raw2).strip()
    if not raw2:
        return dict(term_raw=term_raw, term_clean="", term_lemma="", term_norm="", drop_reason="non_alpha")
    lemmas = []
    for token in tokenize_phrase(raw):
        if ACRONYM_RE.match(token):
            lemmas.append(token)
        elif lemma := nltk_lemma_token(token):
            lemmas.append(lemma)
    if not lemmas:
        return dict(term_raw=term_raw, term_clean=raw, term_lemma="", term_norm="", drop_reason="all_filtered")
    term_lemma = " ".join(lemmas)
    term_norm = smart_title(lemmas)
    return dict(term_raw=term_raw, term_clean=raw, term_lemma=term_lemma,
                term_norm=term_norm if len(term_norm) > 1 else "",
                drop_reason="" if len(term_norm) > 1 else "too_short")

normed = review_term_counts[["term_raw"]].drop_duplicates().copy()
normed_info = normed["term_raw"].apply(normalize_term).apply(pd.Series)
normed = normed.merge(normed_info, on="term_raw", how="left")
term_rows = review_term_counts.merge(
    normed[["term_raw", "term_norm", "term_lemma", "drop_reason"]], on="term_raw", how="left")
term_rows = term_rows[(term_rows["drop_reason"].fillna("") == "") &
                      (term_rows["term_norm"].fillna("") != "")].copy()
vocab = term_rows.groupby("term_norm", as_index=False).agg(
    term_lemma=("term_lemma", lambda values: values.value_counts().index[0]),
    total_count=("count_in_review", "sum"), n_reviews=("review_id", "nunique"),
    n_papers=("paper_code", "nunique"), n_rows=("term_norm", "size"),
    avg_count_per_review=("count_in_review", "mean"),
    n_sent_hit_total=("n_sent_hit", "sum"),
    top_raw_variant=("term_raw", lambda values: values.value_counts().index[0]),
    n_raw_variants=("term_raw", "nunique"))
subject_rows = term_rows[["term_norm", "paper_code"]].drop_duplicates().merge(
    paper_meta[["paper_code", "subject_codes", "year"]], on="paper_code", how="left")
subject_counts = (subject_rows.explode("subject_codes")
                  .dropna(subset=["subject_codes"])
                  .groupby("term_norm")["subject_codes"]
                  .agg(lambda values: len(set(str(value).strip() for value in values if str(value).strip())))
                  .reset_index().rename(columns={"subject_codes": "n_subject_codes"}))
vocab = vocab.merge(subject_counts, on="term_norm", how="left")
vocab["n_subject_codes"] = vocab["n_subject_codes"].fillna(0).astype(int)
vocab = vocab.sort_values(["total_count", "n_reviews", "term_norm"],
                          ascending=[False, False, True]).reset_index(drop=True)
vocab.to_parquet(table_dir / "aspect_vocab.parquet", index=False)
vocab.to_csv(table_dir / "aspect_vocab.csv", index=False, encoding="utf-8-sig")
print(f"Saved {len(vocab)} base vocabulary terms")
